In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from utils.load_data import load_data
from pathlib import Path

In [ ]:
data=load_data('without_absent',2025)

In [ ]:
sns.set_style('white')

In [ ]:
path_output = Path.cwd().parent / 'outputs'

## Math scores distribution (Ukraine overall)

In [ ]:
plt.figure(figsize=(12, 6))
ax = sns.histplot(data['MathBlockBall'], color='darkblue', discrete=True)
for patch in ax.patches:
    if patch.get_x() < 4:
        patch.set_facecolor('darkred')
    else:
        break
plt.axvline(4.5, color='black', linestyle='--')
plt.annotate('Прохідний', (-0.4, 22000), fontsize=17)
plt.title('НМТ 2025. Розподілення балів з математики, вся Україна')
plt.xticks(np.linspace(0, 32, 33))
plt.xlabel('Бали')
plt.ylabel('Кількість')
plt.grid(axis='y')

# Save
plt.savefig(path_output / 'nmt2025_math_ukraine.svg')

plt.show()

In [ ]:
# Skewness

print(data['MathBlockBall'].skew())

#### Analysis

While a naive assumption might expect a normal distribution of test scores, the actual distribution is highly right-skewed (skewness > 1.14), which might indicate that the test was hard for the most participants

Also, the abrupt cutoff and slight elevation of the last bin (32) might indicate that test lack of variance to differentiate between top-tier performers

Distributions for other subjects are similar to this one, though math scores have the highest skewness and one of the highest elevation of the last bin

## Math scores by region

In [ ]:
# Create a copy of the necessary columns
math = data[['RegName','MathBlockBall100']].copy()

# Define the exact boundaries for the intervals
bins = [-np.inf, 99, 120, 140, 160, 180, 200]

# Define the labels for each interval
labels = ['0', '100-120', '120-140', '140-160', '160-180', '180-200']
math['MathBlockBall100'] = pd.cut(math['MathBlockBall100'], bins=bins, labels=labels, right=True)

# Calculate percentages for the stacked bar chart
tab = math.groupby('RegName').value_counts().unstack()
counts = tab.copy()
row_sums = counts.sum(axis=1).replace(0, 1)
perc = counts.div(row_sums, axis=0) * 100

# Plotting
plt.figure(figsize=(14,12))
ax = perc.plot(kind='barh', stacked=True)

ax.legend(
    title='Діапазон балів',
    bbox_to_anchor=(1.05, 1),
    loc='upper left'
)

plt.title('НМТ 2025. Розподілення балів із математики за областями', pad=20)
plt.xlabel('Відсоток')
plt.ylabel('Область')

# Save
plt.savefig(path_output / 'nmt2025_stacked_math_oblasti.svg', bbox_inches='tight')

plt.show()

#### Analysis

Kyiv and the Lviv region demonstrate significantly higher performance compared to the rest of the country. Aside from these, the remaining regions exhibit similar distributions.

## Math score distributions by area (urban/rural)

In [ ]:
nmt_urban_math = data.groupby('TerTypeName')['MathBlockBall'].value_counts()['місто']
nmt_rural_math = data.groupby('TerTypeName')['MathBlockBall'].value_counts()['селище, село']

urban_fraction_math = nmt_urban_math / nmt_urban_math.sum()
rural_fraction_math = nmt_rural_math / nmt_rural_math.sum()

plt.figure(figsize=(12,6))
sns.barplot(urban_fraction_math,alpha=0.6,label='Місто')
sns.barplot(rural_fraction_math,alpha=0.6,label='Селище, село')
plt.legend(title='Тип місцевості')
plt.title('НМТ 2025. Розподіл балів з математики за типом місцевості')
plt.xlabel('Бали')
plt.ylabel('Частка учасників')
plt.grid(axis='y')

# Save
plt.savefig(path_output / "nmt2025_math_city-rural.svg", bbox_inches='tight')

plt.show()

In [ ]:
# Proportion of perfect scores in urban and rural areas

print(urban_fraction_math[32] / rural_fraction_math[32])

#### Analysis

Rural areas demonstrate significantly lower overall performance compared to urban centers, featuring a higher proportion of low-scoring participants (1-9) and fewer high-scoring participants (10-32). Notably, the proportion of perfect scores (32) in urban areas is roughly five times greater than in rural areas.

## Math scores distribution (Kyiv)

In [ ]:
plt.figure(figsize=(12,6))
ax=sns.histplot(data[data['AreaName']=='м.Київ']['MathBlockBall'],bins=34,color='darkblue',discrete=True)
for patch in ax.patches:
    if patch.get_x() < 4:
        patch.set_facecolor('darkred')
    else:
        break
plt.axvline(4.5,color='black',linestyle='--')
plt.title("НМТ 2025. Розподілення балів з математики, м. Київ")
plt.xticks(np.linspace(0,32,33))
plt.annotate('Прохідний',(0.5,2400),fontsize=12)
plt.xlabel('Бали')
plt.ylabel('Кількість')
plt.grid(axis='y')

# Save
plt.savefig(path_output / "nmt2025_math_kyiv.svg")

plt.show()

In [ ]:
# Skewness

data[data['AreaName']=='м.Київ']['MathBlockBall'].skew()

#### Analysis

Consistent with the regional breakdown, the distribution for Kyiv exhibits a notably lower skewness (0.88) compared to the national average (1.14), reflecting a higher concentration of successful test-takers.

## Number of participants with the highest scores by subject

In [ ]:
ball200=(data[['UkrBlockBall100','MathBlockBall100','HistBlockBall100','PhysBlockBall100','ChemBlockBall100','BioBlockBall100','GeoBlockBall100','EngBlockBall100','FraBlockBall100','DeuBlockBall100','SpaBlockBall100','UkrLitBlockBall100']]==200).sum()
ball200.index=['Українська мова','Математика','Історія України','Фізика','Хімія','Біологія','Географія','Англійська мова','Французька мова','Німецька мова','Іспанська мова','Українська література']
ball200.sort_values(ascending=False,inplace=True)

plt.figure(figsize=(14,6))
sns.barplot(ball200,orient='h')
plt.title('НМТ 2025. Кількість двохсотбальних результатів за предметами',pad=10)

# Save
plt.savefig(path_output / 'nmt2025_highest-scores-by-subject.svg')

plt.show()

#### Analysis

Because Mathematics, Ukrainian Language, and History of Ukraine are mandatory for all participants, their baseline sample sizes are identical. The fact that perfect scores in mathematics eclipse those in history and language by such a massive margin confirms a flaw in the test design. The mathematics exam lacked the necessary variance and difficulty at the upper extreme

English language ranks second, which is highly notable given it is an optional subject with a smaller overall participant pool than the mandatory. This high volume of perfect scores suggests that the subset of students who elect to take English are already highly proficient

## Correlation of mandatory subjects with non-mandatory

In [ ]:
# Define subjects
mandatory = ['UkrBlockBall', 'MathBlockBall', 'HistBlockBall']
optional = ['PhysBlockBall', 'ChemBlockBall', 'BioBlockBall', 'GeoBlockBall', 'EngBlockBall', 'UkrLitBlockBall']
all_subjects = mandatory + optional

# Replace -1 (did not take) with NaN so they don't skew the correlation, then calculate
corr_matrix = data[all_subjects].replace(-1, np.nan).corr()

# Slice the matrix: Rows = Mandatory, Columns = All Subjects
heat_all = corr_matrix.loc[mandatory, all_subjects]

# Rename the indices and columns for the plot
heat_all.index = ['Українська мова', 'Математика', 'Історія України']
heat_all.columns = ['Укр. мова', 'Математика', 'Історія Укр.', 'Фізика', 'Хімія', 'Біологія', 'Географія', 'Англ. мова', 'Укр. літ.']

# Create a simple mask to hide the lower triangle of the mandatory subjects
mask = np.zeros_like(heat_all, dtype=bool)
mask[1, 0] = True  # Hide Math vs Ukr
mask[2, 0] = True  # Hide Hist vs Ukr
mask[2, 1] = True  # Hide Hist vs Math

# Plotting
plt.figure(figsize=(16,6))
sns.set_context("notebook", font_scale=1.4)
sns.heatmap(heat_all,annot=True,mask=mask,vmin=0.5,cbar_kws={'pad': 0.12},cmap='mako')
plt.title("НМТ 2025. Кореляція між предметами",pad=70,fontsize=18,)
ax=plt.gca()
ax.xaxis.tick_top()
ax.yaxis.tick_right()
plt.yticks(rotation=0)
plt.tight_layout()
plt.grid(False)

# Save
plt.savefig(path_output / "nmt2025_corrs.svg")

plt.show()

#### Analysis

All subjects exhibit a moderate to strong positive correlation with one another, indicating that general academic aptitude translates across disciplines. However, distinct disciplinary clusters emerge upon closer inspection:

Mathematics serves as a highly accurate predictor for success in hard sciences, showing the strongest overall correlations in the dataset with Physics (0.79) and Chemistry (0.76)

A similar domain-specific relationship exists within the humanities. Both Ukrainian Language and History of Ukraine correlate highly with Ukrainian Literature (0.75)

English Language demonstrates the weakest correlation with the mandatory subjects (ranging from 0.54 to 0.59). This noticeable divergence implies that English proficiency is somewhat independent of the standard academic core, potentially driven by external factors

## Distribution of educational organizations

In [ ]:
palette1=['#BF616A', '#D08770', '#EBCB8B', '#8FBCBB', '#88C0D0', '#81A1C1', '#5E81AC']

EOType_count=data.groupby('EOTypeName')['MathBlockBall'].count()
EO_Type=pd.concat([EOType_count[EOType_count > 4000 ],pd.Series({'Інше':EOType_count[EOType_count < 4000].sum()})]).sort_values()

name_map={'заклад фахової передвищої освіти':'Заклад фахової передвищої освіти','спеціалізована школа':'Спеціалізована школа','навчально-виховний комплекс':'Навчально-виховний комплекс','заклад професійної (професійно-технічної) освіти':'Заклад професійної освіти','середня загальноосвітня школа':'Середня загальноосвітня школа','ліцей':'Ліцей'}
labels = [name_map.get(name, name) for name in EO_Type.index]

# Plotting
plt.pie(EO_Type[::-1],startangle=90,radius=1.5,colors=palette1[::-1],counterclock=False,autopct='%1.1f%%',pctdistance=1.25,textprops={'fontsize': 14, 'weight': 'bold'})
plt.legend(labels[::-1],bbox_to_anchor=(2.2, 0.5),loc='center')
plt.title("Розподіл учасників за типом закладу освіти",pad=110,fontsize=20)

# Save
plt.savefig(path_output / "nmt2025_EOType.svg",bbox_inches='tight')

plt.show()

#### Analysis

The distribution of test participants by educational institution heavily reflects the ongoing structural reforms within the Ukrainian educational system.

Over half of all participants (54.3%) are registered at Lyceums, which is the result of recent national educational legislation. Due to the same reason, the share of standard secondary schools (14.8%) will decrease further

Institutions of professional pre-higher education (colleges) represent the second-largest demographic at 19.3%. This highlights a significant cohort of students who chose a route after the 9th grade and are now taking the standardized exam to transition into full Bachelor's degree programs at universities.

## Math scores distributions by educational organization type

In [ ]:
# Plotting
plt.figure(figsize=(18,7))
sns.set_context("notebook", font_scale=1.7)
violin_data=data[['MathBlockBall','EOTypeName']][data['EOTypeName'].isin(EO_Type.drop('Інше').index.values)].copy()
violin_data['EOTypeName']=violin_data['EOTypeName'].replace(name_map)
sns.violinplot(violin_data,x='EOTypeName',y='MathBlockBall',palette='terrain',cut=0)
plt.ylabel('Бал',fontsize=18)
plt.xlabel('')
plt.yticks(np.linspace(0,32,9))
plt.ylim(-1,33)
plt.xticks(fontsize=18,rotation=6)
ax=plt.gca()
ax.yaxis.grid(True)
plt.title("Розподіл балів з математики за типом закладу освіти",fontsize=20,pad=20)
for spine in ax.spines.values():
    spine.set_color('black')
plt.tight_layout()

# Save
plt.savefig(path_output / "nmt2025_violins.svg",bbox_inches='tight')

plt.show()

#### Analysis

The distribution of mathematics scores varies drastically across different types of educational institutions. Students from Specialized Schools (Спеціалізована школа) demonstrate the strongest overall performance (highest median and widest density in upper quartiles). Lyceums (Ліцей), Educational-Upbringing Complexes (НВК) and standard secondary school (СЗШ) have similar shape, representing the national baseline. Vocational Schools (Заклад професійної освіти) and Pre-Higher Educational Institutions (Заклад фахової передвищої освіти) display the weakest mathematical performance.

## Joint distribution of mathematics and physics

In [ ]:
# Plotting
plt.figure(figsize=(10,10))
sns.set_context("notebook", font_scale=1.5)
sns.set_style('ticks')
sns.jointplot(data[data['PhysBlockBall']!=-1][['MathBlockBall','PhysBlockBall']],x='MathBlockBall',y='PhysBlockBall',marginal_kws={'bins':33},kind='hex',color='royalblue',ratio=4,marginal_ticks=True)
plt.tight_layout()
plt.grid(False)
plt.ylabel("Фізика",fontsize=18)
plt.xlabel("Математика",fontsize=18)
plt.yticks(np.linspace(0,32,9))
plt.xticks(np.linspace(0,32,9))
plt.title('Розподіл балів з фізики та математики',pad=130,fontsize=20)
ax=plt.gca()

# Save
plt.savefig(path_output / "nmt2025_phys_math_joint.svg",bbox_inches='tight')

plt.show()

#### Analysis

It is noticeable that math and physics have a strong positive correlation

The highest concentration of participants sits in the lower-left quadrant, which indicates that the vast majority of students taking both exams struggled symmetrically across both disciplines

There is an asymmetry between the top-left and bottom-right quadrants. The top-left region is empty, which demonstrates that mathematical proficiency acts as a strict prerequisite for success in physics

The marginal histograms confirm that both subjects suffer from a ceiling effect. Despite the heavy right-skewness indicating high overall difficulty, both distributions show an unnatural spike at the absolute maximum score of 32, once again highlighting a lack of test headroom for top-tier performers.